# Gas Giant Generator

This notebook uses the `SphereCamera` class from `render/camera.py` to render gas giants with rings.

In [ ]:
# Environment Setup
import os
import sys
import subprocess

# 1. Handle Colab/Local Filesystem and Cloning
IS_COLAB = 'google.colab' in sys.modules
REPO_URL = "https://github.com/pmcray/cosmic_engine.git"
PROJECT_NAME = "cosmic_engine"

def setup_environment():
    if IS_COLAB:
        print("Running on Google Colab.")
        if not os.path.exists(f"/content/{PROJECT_NAME}"):
            print(f"Cloning {PROJECT_NAME} repository...")
            subprocess.run(["git", "clone", REPO_URL], check=True)
        
        # Change to project root
        os.chdir(f"/content/{PROJECT_NAME}")
        sys.path.append(os.getcwd())
    else:
        # Local setup: move up from /notebooks to the project root
        current_dir = os.getcwd()
        if os.path.basename(current_dir) == "notebooks":
            root_dir = os.path.abspath(os.path.join(current_dir, ".."))
        else:
            root_dir = current_dir
        
        if root_dir not in sys.path:
            sys.path.append(root_dir)
        os.chdir(root_dir)
        print(f"Local root added to sys.path: {root_dir}")

setup_environment()

# 2. Dependency Management
try:
    import taichi as ti
    import numpy as np
    from PIL import Image as PILImage
    print("Dependencies already installed.")
except ImportError:
    print("Installing dependencies...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "taichi", "numpy", "Pillow"], check=True)
    import taichi as ti
    import numpy as np
    from PIL import Image as PILImage

print(f"Current working directory: {os.getcwd()}")
print(f"Taichi version: {ti.__version__}")

In [ ]:
import taichi as ti
import numpy as np
from render.camera import SphereCamera
from physics.fluid_solver import FluidEngine
from IPython.display import display, Image
import PIL.Image

# 1. Select Planet Type
# Options: "jupiter", "saturn", "uranus", "venus", "hot_jupiter"
PLANET_TYPE = "jupiter"
RES = 1024
HAS_RINGS = 1 # 1 for rings, 0 for no rings

# 2. Initialize Engines
ti.init(arch=ti.gpu)

fluid = FluidEngine(res=RES, planet_type=PLANET_TYPE)
camera = SphereCamera(fluid_res=RES, render_res=RES, has_rings=HAS_RINGS, samples=4)

# 3. Run Fluid Simulation to establish atmosphere
SIM_FRAMES = 200
print(f"Simulating {PLANET_TYPE} atmosphere for {SIM_FRAMES} frames...")
for f in range(SIM_FRAMES):
    fluid.step(f)
    if f % 50 == 0:
        print(f"  Frame {f}/{SIM_FRAMES}")

# 4. Render 3D Gas Giant
print("\nRendering final output (this may take a moment at high resolution)...")
camera.render_gas_giant(fluid.dye, 0.5, 0.3, -1.0)

img = camera.get_image_data()
display(PIL.Image.fromarray((img * 255).astype(np.uint8)))
print("Done!")